# Auchan scrap
## Récuperation des IDs des magasins

In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import os

def scroll_to_bottom(driver):
    """Scroll to the bottom of the page until no new content loads"""
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        # Scroll down
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        
        # Wait for new content to load
        time.sleep(2)
        
        # Calculate new scroll height
        new_height = driver.execute_script("return document.body.scrollHeight")
        
        # Break if no new content loaded
        if new_height == last_height:
            break
        last_height = new_height

def scrape_auchan_stores(output_path):
    """
    Scrape Auchan store data and save to specified path
    
    Args:
        output_path (str): Full path where to save the JSON file
    """
    # Initialize the list to store store data
    stores_data = {
        "stores": []
    }
    
    # Initialize undetected-selenium
    options = uc.ChromeOptions()
    options.add_argument('--headless')  # Run in headless mode
    driver = uc.Chrome(options=options)
    
    try:
        # Navigate to the Auchan Drive stores page
        driver.get("https://www.auchan.fr/nos-magasins?types=DRIVE")
        
        # Wait for the content to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "store-list__department-name"))
        )
        
        # Scroll to load all content
        scroll_to_bottom(driver)
        
        # Find all departments
        departments = driver.find_elements(By.CLASS_NAME, "store-list__department-name")
        
        for department in departments:
            department_name = department.text
            
            # Find the parent element containing stores for this department
            department_container = department.find_element(By.XPATH, "./following-sibling::ul")
            store_containers = department_container.find_elements(By.CLASS_NAME, "place-pos__main-infos")
            
            # Process each store in the department
            for store in store_containers:
                try:
                    # Get store type (Drive)
                    store_type = store.find_element(By.CLASS_NAME, "place-pos__type-name").text
                    
                    # Get store name
                    store_name = store.find_element(By.CLASS_NAME, "place-pos__name").text
                    
                    # Get address
                    address = store.find_element(By.CLASS_NAME, "place-pos__address").find_element(By.TAG_NAME, "span").text
                    postal_code = address.split()[0]  # Extract postal code
                    city = ' '.join(address.split()[1:])  # Extract city
                    
                    # Create store object
                    store_object = {
                        "department": department_name,
                        "name": store_name,
                        "type": store_type,
                        "postal_code": postal_code,
                        "city": city,
                        "full_address": address
                    }
                    
                    # Add to stores list
                    stores_data["stores"].append(store_object)
                    
                except Exception as e:
                    print(f"Error processing store: {e}")
                    continue
        
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        # Save the data to a JSON file
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(stores_data, f, ensure_ascii=False, indent=4)
            
        print(f"Data successfully saved to: {output_path}")
        return stores_data
        
    finally:
        driver.quit()

if __name__ == "__main__":
    # Specify your desired output path here
    output_file = os.path.join(os.path.expanduser("~"), "Documents", "auchan_stores.json")
    
    stores = scrape_auchan_stores(output_file)
    print(f"Successfully scraped {len(stores['stores'])} stores")
    print("Sample of stores:", json.dumps(stores['stores'][:2], indent=4, ensure_ascii=False))

Permettras d'iterer sur plusieurs magasins si besoin 

## Scrapping (liens catégories, liens produits)

### Liens catégories 

In [ ]:
##extraire les liens des catégories 
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import os

# Initialize undetected ChromeDriver
driver = uc.Chrome()

try:
    # Open the Auchan website
    driver.get("https://www.auchan.fr/")
    
    # Wait for the "Accepter et fermer" button and click it
    try:
        accept_cookies_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accepter et fermer')]"))
        )
        accept_cookies_button.click()
        print("Cookies accepted.")
    except Exception as e:
        print("No cookies popup found or already accepted:", e)
    
    # Click on the target button
    try:
        target_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "/html/body/div[3]/header/div[2]/div/button"))
        )
        target_button.click()
        print("Target button clicked.")
    except Exception as e:
        print("Target button not found or not clickable:", e)
    
    # Extract div elements with the specified class and data attributes
    links_dict = {}
    try:
        navigation_nodes = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.XPATH, "//div[contains(@class, 'navigation-node-sorted navigationNode')]"))
        )
        for node in navigation_nodes:
            node_title = node.get_attribute("data-node-title")
            link_element = node.find_element(By.TAG_NAME, "a")
            href = link_element.get_attribute("href")
            links_dict[node_title] = href
            
        # Save the dictionary to a JSON file
        output_file = "auchan_links.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(links_dict, f, ensure_ascii=False, indent=4)
        print(f"Data successfully saved to {output_file}")
        
        # Print the dictionary
        print("\nExtracted links and titles:")
        for title, link in links_dict.items():
            print(f"  - {title}: {link}")
            
    except Exception as e:
        print(f"Error extracting links and titles: {e}")

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # Close the browser
    driver.quit()

(possibilitées de trier links_dict si des catégories ne sont pas pertinentes )

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import json
from urllib.parse import urljoin
import os

def load_existing_data():
    """Load existing data from JSON file if it exists"""
    if os.path.exists('auchan_products.json'):
        try:
            with open('auchan_products.json', 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"Error loading existing data: {e}")
    return {}

def save_category_data(category_product_links):
    """Save the current state of the data to JSON file"""
    try:
        with open('auchan_products.json', 'w', encoding='utf-8') as f:
            json.dump(category_product_links, f, ensure_ascii=False, indent=4)
        print("Data saved successfully")
    except Exception as e:
        print(f"Error saving data: {e}")

# Load any existing data
category_product_links = load_existing_data()

# Headers to mimic a real browser
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Iterate over categories and scrape product links
for category_name, category_link in links_dict.items():
    # Skip if category already scraped
    if category_name in category_product_links:
        print(f"Category {category_name} already scraped, skipping...")
        continue
        
    print(f"\nScraping category: {category_name}")
    page_number = 1
    category_dict = {}  # Dictionary to store product info for this category
    
    try:
        while True:
            paginated_link = f"{category_link}?page={page_number}"
            print(f"Scraping page: {paginated_link}")
            
            try:
                response = requests.get(paginated_link, headers=headers, timeout=10)
                response.raise_for_status()
            except requests.exceptions.RequestException as e:
                print(f"Error requesting page {page_number}: {e}")
                break
                
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Determine the last page number if it's the first page
            if page_number == 1:
                try:
                    pagination = soup.select_one("nav.pagination-main__container")
                    if pagination:
                        last_page_element = pagination.select("div.pagination-links__container a")[-1]
                        last_page = int(last_page_element.text.strip())
                        print(f"Last page for {category_name}: {last_page}")
                    else:
                        print("Single page category")
                        last_page = 1
                except Exception as e:
                    print(f"Error determining last page: {e}")
                    last_page = 1
            
            # Locate product links and names
            product_articles = soup.select("article.product-thumbnail")
            
            # If no products found, break the loop
            if not product_articles:
                print("No more products found on this page.")
                break
            
            # Process each product
            for article in product_articles:
                try:
                    link_element = article.select_one("div a")
                    if link_element and 'href' in link_element.attrs:
                        product_link = urljoin(category_link, link_element['href'])
                        
                        # Try to get the product name
                        name_element = article.select_one("h3")  # Adjust selector based on actual HTML
                        product_name = name_element.text.strip() if name_element else "Unknown Product"
                        
                        # Store in category dictionary with name as key and link as value
                        category_dict[product_name] = product_link
                        
                except Exception as e:
                    print(f"Error processing product: {e}")
                    continue
            
            print(f"Found {len(product_articles)} products on page {page_number}")
            
            if page_number >= last_page:
                print("Reached the last page")
                break
                
            page_number += 1
            time.sleep(2)  # Anti-ban delay
            
    except Exception as e:
        print(f"Error processing category {category_name}: {e}")
    
    # Store category dictionary in main dictionary
    category_product_links[category_name] = category_dict
    print(f"Total products in {category_name}: {len(category_dict)}")
    
    # Save after each category is completed
    save_category_data(category_product_links)

# Print final summary
print("\nScraped Products Summary:")
for category, products in category_product_links.items():
    print(f"{category}: {len(products)} products")
    print("Sample products:")
    # Print first 3 products as example
    for name, link in list(products.items())[:3]:
        print(f"  - {name}: {link}")

Le code ci dessous prends la liste des catégories , itere sur toutes ses page puis enregistre un json avec le nom de catégorie associer au lien d'un produit 

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import json

# Headers to simuler un vrai navigateur
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Créer un dictionnaire pour stocker les liens produits
category_product_links = {}

# Process only the first category
category_name, category_link = next(iter(links_dict.items()))
print(f"\nScraping category: {category_name}")
page_number = 1
category_links_list = []

while True:
    # Ajoute le paramètre de pagination
    paginated_link = f"{category_link}?page={page_number}"
    print(f"Scraping page: {paginated_link}")

    response = requests.get(paginated_link, headers=headers)
    if response.status_code != 200:
        print(f"Failed to retrieve page {page_number} for category {category_name}. Status code: {response.status_code}")
        break

    soup = BeautifulSoup(response.text, 'html.parser')

    # Déterminer le nombre de pages
    if page_number == 1:
        try:
            last_page_element = soup.select_one("nav.pagination-main__container div.pagination-links__container a:nth-last-child(1)")
            if last_page_element and last_page_element.text.isdigit():
                last_page = int(last_page_element.text.strip())
                print(f"Last page for {category_name}: {last_page}")
            else:
                print("Could not determine the last page. Exiting loop.")
                break
        except Exception as e:
            print(f"Error determining the last page: {e}")
            break

    # Localiser les produits
    product_articles = soup.select("article.product-thumbnail div a")
    if not product_articles:
        print("No more products found on this page. Exiting loop.")
        break

    # Extraire les liens et les compléter
    base_url = "https://www.auchan.fr"
    page_product_links = [
        base_url + article['href'] 
        for article in product_articles 
        if 'href' in article.attrs
    ]

    category_links_list.extend(page_product_links)
    print(f"Found {len(page_product_links)} products on page {page_number}.")

    if page_number >= last_page:
        print("Reached the last page. Exiting loop.")
        break

    page_number += 1
    time.sleep(2)

# Stocker les liens sous la catégorie
category_product_links[category_name] = category_links_list
print(f"\nTotal products in {category_name}: {len(category_links_list)}")

# Enregistrer dans un fichier JSON
with open("auchan_products_links.json", "w", encoding="utf-8") as f:
    json.dump(category_product_links, f, indent=2, ensure_ascii=False)

print("\nFichier auchan_products_links.json sauvegardé avec succès !")

2 options de scrapping
Besoin de mettre la liste de lien créer precedement en entrée et besoin de revoir les disponibilitées 

In [ ]:
import csv

class AuchanScraper:

    def __init__(self):
        self.driver = Chrome()
        self.store_selected = False
        self.cookies_accepted = False
        self.all_feature_titles = set()
        self.output_file = "products_data_live.csv"

        # Crée le fichier CSV avec les colonnes nécessaires
        if not os.path.exists(self.output_file):
            with open(self.output_file, mode='w', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow(["URL", "Brand", "Product Name", "Category", "Price", "Rating", "Availability"])

    def handle_cookies(self):
        """Handle cookie acceptance only once"""
        if not self.cookies_accepted:
            try:
                accept_cookies_button = WebDriverWait(self.driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accepter et fermer')]"))
                )
                accept_cookies_button.click()
                print("Cookies accepted.")
                self.cookies_accepted = True
            except:
                print("No cookies popup found or already accepted")
                self.cookies_accepted = True

    def select_store(self):
        """Handle store selection only once"""
        if not self.store_selected:
            try:
                # Click the initial button for store selection
                button = WebDriverWait(self.driver, 5).until(
                    EC.presence_of_element_located((By.XPATH, 
                        "/html/body/div[3]/div[2]/div[2]/div[4]/div[1]/div[2]/div/div[4]/button"))
                )
                button.click()
                print("Store selection button clicked")

                # Handle postal code input
                input_field = WebDriverWait(self.driver, 5).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "input.journey__search-input"))
                )
                
                input_field.clear()
                input_field.click()
                input_field.send_keys("75000")
                print("Postal code entered: 75000")
                
                time.sleep(1.5)
                
                # Select first suggestion
                actions = ActionChains(self.driver)
                actions.send_keys(Keys.ARROW_DOWN).send_keys(Keys.RETURN).perform()
                print("First suggestion selected")
                
                time.sleep(1)

                # Click the first store's button
                first_store = WebDriverWait(self.driver, 5).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "button.btnJourneySubmit"))
                )
                first_store.click()
                print("Store selected successfully")
                
                self.store_selected = True
                time.sleep(2)
                
            except Exception as e:
                print(f"Error during store selection: {e}")
                return False
        return True

    def save_to_csv(self, product_info):
        """Sauvegarde les informations produit dans le fichier CSV."""
        try:
            with open(self.output_file, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([
                    product_info.get("URL", "N/A"),
                    product_info.get("Brand", "N/A"),
                    product_info.get("Product Name", "N/A"),
                    product_info.get("Category", "N/A"),
                    product_info.get("Price", "N/A"),
                    product_info.get("Rating", "N/A"),
                    product_info.get("Availability", "N/A"),
                ])
            print(f"Saved to CSV: {product_info.get('Product Name', 'Unknown Product')}")
        except Exception as e:
            print(f"Error saving to CSV: {e}")

    def process_product(self, url, category):
        """Traite un produit unique."""
        try:
            self.driver.get(url)
            print(f"\nProcessing URL: {url}")

            self.handle_cookies()
            if not self.select_store():
                return None

            # Extraire les informations produit
            product_info = self.extract_product_data()
            if product_info:
                product_info["Category"] = category  # Ajoute la catégorie au produit
                self.save_to_csv(product_info)  # Enregistre immédiatement dans le CSV
                return product_info

        except Exception as e:
            print(f"Error processing URL {url}: {e}")
            return None

    def process_products(self, urls, category):
        """Traite plusieurs URLs de produits."""
        for url in urls:
            self.process_product(url, category)

    def extract_product_data(self):
        """Extrait les données produit nécessaires."""
        try:
            soup = BeautifulSoup(self.driver.page_source, 'html.parser')
            product_info = {"URL": self.driver.current_url}

            # Extrait les données principales
            product_info["Brand"] = soup.find('span', {'class': 'brand'}).text.strip() if soup.find('span', {'class': 'brand'}) else "N/A"
            product_info["Product Name"] = soup.find('h1').text.strip() if soup.find('h1') else "N/A"
            product_info["Price"] = soup.find('span', {'class': 'price'}).text.strip() if soup.find('span', {'class': 'price'}) else "N/A"
            product_info["Rating"] = soup.find('meta', {'itemprop': 'ratingValue'})['content'] if soup.find('meta', {'itemprop': 'ratingValue'}) else "N/A"
            product_info["Availability"] = self.check_availability()

            return product_info

        except Exception as e:
            print(f"Error extracting product data: {e}")
            return None

    def check_availability(self):
        """Vérifie la disponibilité du produit."""
        try:
            availability = self.driver.find_element(By.CSS_SELECTOR, 'meta[itemprop="availability"]')
            if "InStock" in availability.get_attribute("content"):
                return "Available"
            elif "OutOfStock" in availability.get_attribute("content"):
                return "Not available"
        except:
            pass
        return "Status unknown"

    def run(self, category_urls):
        """Lance le scraping pour chaque catégorie."""
        for category, urls in category_urls.items():
            print(f"Processing category: {category}")
            self.process_products(urls, category)
        self.driver.quit()


# Exemple d'utilisation
if __name__ == "__main__":


    scraper = AuchanScraper()
    scraper.run(updated_text_content)

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from undetected_chromedriver import Chrome
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from datetime import datetime

class OptimizedAuchanScraper:
    def __init__(self):
        # Configuration du navigateur pour optimiser les performances
        options = webdriver.ChromeOptions()
        options.add_argument('--disable-gpu')
        options.add_argument('--disable-extensions')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-logging')
        
        self.driver = Chrome(options=options)
        self.store_selected = False
        self.cookies_accepted = False
        
        # Création du fichier de suivi avec horodatage
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.csv_filename = f"auchan_products_{self.timestamp}.csv"
        
        # Initialisation du fichier CSV avec les colonnes nécessaires
        if not os.path.exists(self.csv_filename):
            pd.DataFrame(columns=[
                "URL", "Brand", "Product Name", "Rating", 
                "Availability", "Category", "Timestamp"
            ]).to_csv(self.csv_filename, index=False)

    def handle_cookies(self):
        """Gestion des cookies avec la logique exacte du site Auchan"""
        if not self.cookies_accepted:
            try:
                accept_cookies_button = WebDriverWait(self.driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accepter et fermer')]"))
                )
                accept_cookies_button.click()
                print("Cookies accepted.")
                self.cookies_accepted = True
            except:
                print("No cookies popup found or already accepted")
                self.cookies_accepted = True
    def select_store(self):
        """Handle store selection only once"""
        if not self.store_selected:
            try:
                # Click the initial button for store selection
                button = WebDriverWait(self.driver, 5).until(
                    EC.presence_of_element_located((By.XPATH, 
                        "/html/body/div[3]/div[2]/div[2]/div[4]/div[1]/div[2]/div/div[4]/button"))
                )
                button.click()
                print("Store selection button clicked")

                # Handle postal code input
                input_field = WebDriverWait(self.driver, 5).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "input.journey__search-input"))
                )
                
                input_field.clear()
                input_field.click()
                input_field.send_keys("75000")
                print("Postal code entered: 75000")
                
                time.sleep(1.5)
                
                # Select first suggestion
                actions = ActionChains(self.driver)
                actions.send_keys(Keys.ARROW_DOWN).send_keys(Keys.RETURN).perform()
                print("First suggestion selected")
                
                time.sleep(1)

                # Click the first store's button
                first_store = WebDriverWait(self.driver, 5).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "button.btnJourneySubmit"))
                )
                first_store.click()
                print("Store selected successfully")
                
                self.store_selected = True
                time.sleep(2)
                
            except Exception as e:
                print(f"Error during store selection: {e}")
                return False
        return True

    def extract_essential_data(self):
        """
        Extrait les données essentielles d'un produit en utilisant les sélecteurs
        qui ont fait leurs preuves dans le code original.
        """
        try:
            soup = BeautifulSoup(self.driver.page_source, 'html.parser')
            
            product_info = {
                "URL": self.driver.current_url,
                "Brand": "N/A",
                "Product Name": "N/A",
                "Price": "N/A",
                "Price per Unit": "N/A",
                "Rating": "N/A",
                "Availability": "N/A"
            }

            # Extraction de la marque - utilise le même XPath que le code original
            try:
                brand = self.driver.find_element(
                    By.XPATH, 
                    "/html/body/div[3]/div[2]/div[2]/div[4]/div[1]/div[2]/div/div[2]/div/div[1]/a/bold"
                )
                product_info["Brand"] = brand.text.strip()
            except Exception as e:
                print(f"Erreur lors de l'extraction de la marque: {e}")

            # Extraction du nom - utilise le même XPath que le code original
            try:
                name = self.driver.find_element(
                    By.XPATH, 
                    "/html/body/div[3]/div[2]/div[2]/div[4]/div[1]/div[2]/div/div[2]/div/div[1]/h1"
                )
                product_info["Product Name"] = name.text.strip()
            except Exception as e:
                print(f"Erreur lors de l'extraction du nom: {e}")

            # Extraction du prix - utilise les mêmes sélecteurs CSS que le code original
            try:
                price_selectors = [
                    'div.product-price.product-price--large.bolder.text-dark-color',
                    'div.product-price--large.bolder.text-dark-color',
                    'div.product-price--large',
                    'meta[itemprop="price"]'
                ]
                
                price = None
                for selector in price_selectors:
                    price_element = soup.select_one(selector)
                    if price_element:
                        if selector == 'meta[itemprop="price"]':
                            price = f"{price_element.get('content')}€"
                        else:
                            price = price_element.text.strip()
                        break
                
                product_info["Price"] = price if price else "N/A"

                # Extraction du prix par unité - utilise les mêmes sélecteurs que le code original
                price_unit_selectors = [
                    'div.product-price--smaller > span',
                    'div.product-price--smaller span',
                    'div.offer-selector__price-container span:first-child'
                ]
                
                price_unit = None
                for selector in price_unit_selectors:
                    price_unit_element = soup.select_one(selector)
                    if price_unit_element and price_unit_element.text.strip():
                        price_unit = price_unit_element.text.strip()
                        break

                product_info["Price per Unit"] = price_unit if price_unit else "N/A"

            except Exception as e:
                print(f"Erreur lors de l'extraction du prix: {e}")

            # Extraction de la note
            try:
                rating = soup.find('meta', {'itemprop': 'ratingValue'})
                if rating:
                    product_info["Rating"] = rating['content']
            except Exception as e:
                print(f"Erreur lors de l'extraction de la note: {e}")

            # Vérification de la disponibilité
            product_info["Availability"] = self.check_availability()

            # Affichage des données extraites pour vérification
            print("\nDonnées extraites :")
            for key, value in product_info.items():
                print(f"{key}: {value}")

            return product_info

        except Exception as e:
            print(f"Erreur générale lors de l'extraction: {e}")
            return None
    def check_availability(self):
        """
        Vérifie la disponibilité du produit en utilisant plusieurs méthodes.
        Returns: 
            str: "Available", "Almost there", or "Not available"
        """
        try:
            # Vérification du statut "Non disponible"
            try:
                not_available_msg = self.driver.find_element(
                    By.CSS_SELECTOR, 
                    "div.product-unavailable__message"
                )
                if not_available_msg and "n'est plus dans notre gamme" in not_available_msg.text:
                    return "Not available"
            except:
                pass

            # Vérification du statut "Bientôt disponible"
            try:
                bientot_element = self.driver.find_element(
                    By.CSS_SELECTOR,
                    "div.product-unavailable__message--large"
                )
                if bientot_element and "Bientôt dispo" in bientot_element.text:
                    return "Almost there"
            except:
                pass

            try:
                out_of_stock = self.driver.find_element(
                    By.CSS_SELECTOR,
                    'meta[itemprop="availability"][content="https://schema.org/OutOfStock"]'
                )
                if out_of_stock:
                    return "Almost there"
            except:
                pass

            # Vérification du statut "Disponible"
            try:
                add_to_cart = self.driver.find_element(
                    By.CSS_SELECTOR,
                    "div.offer-selector__action-wrapper--add-to-cart"
                )
                if add_to_cart:
                    return "Available"
            except:
                pass

            try:
                in_stock = self.driver.find_element(
                    By.CSS_SELECTOR,
                    'meta[itemprop="availability"][content="https://schema.org/InStock"]'
                )
                if in_stock:
                    return "Available"
            except:
                pass

            return "Status unknown"

        except Exception as e:
            print(f"Error checking availability: {e}")
            return "Status unknown"
    def save_product_data(self, product_info, url):
        """Sauvegarde les données d'un produit dans le CSV"""
        if product_info:
            product_info["URL"] = url
            product_info["Timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
            df_new = pd.DataFrame([product_info])
            df_new.to_csv(self.csv_filename, mode='a', header=False, index=False)
            
            print(f"Données sauvegardées pour : {product_info.get('Product Name', 'Produit inconnu')}")
            return True
        return False

    def process_product(self, url, category):
        """Traitement d'un produit individuel"""
        try:
            self.driver.get(url)
            self.handle_cookies()
            if not self.select_store():
                return False

            product_info = self.extract_essential_data()
            if product_info:
                product_info["Category"] = category
                return self.save_product_data(product_info, url)
            
        except Exception as e:
            print(f"Erreur lors du traitement de l'URL {url}: {e}")
        return False
    def save_product_data(self, product_info, url):
 
        if product_info:
            # Ajout des informations de traçabilité
            product_info["URL"] = url
            product_info["Timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
            # Conversion en DataFrame et sauvegarde
            df_new = pd.DataFrame([product_info])
            df_new.to_csv(self.csv_filename, mode='a', header=False, index=False)
            
            # Affichage des informations qui viennent d'être sauvegardées
            print("\n=== Informations enregistrées pour ce produit ===")
            print(f"Marque: {product_info.get('Brand', 'N/A')}")
            print(f"Nom du produit: {product_info.get('Product Name', 'N/A')}")
            print(f"Note: {product_info.get('Rating', 'N/A')}")
            print(f"Disponibilité: {product_info.get('Availability', 'N/A')}")
            print(f"Catégorie: {product_info.get('Category', 'N/A')}")
            print(f"URL: {product_info.get('URL', 'N/A')}")
            print(f"Horodatage: {product_info.get('Timestamp', 'N/A')}")
            print("============================================\n")
            
            # Affichage des dernières entrées du CSV
            try:
                print("\n=== Aperçu des 5 dernières entrées du CSV ===")
                df_all = pd.read_csv(self.csv_filename)
                print(df_all.tail().to_string())
                print("============================================\n")
            except Exception as e:
                print(f"Erreur lors de la lecture du CSV: {e}")
            
            return True
        return False

def main():
    """
    Fonction principale qui gère le processus de scraping en utilisant updated_text_content.
    Cette fonction traite les URLs organisées par catégories et maintient un suivi détaillé
    de la progression.
    """
    # Initialisation du scraper
    scraper = OptimizedAuchanScraper()
    processed_count = 0
    total_products = sum(len(urls) for urls in updated_text_content.values())
    
    try:
        # Affichage initial pour le suivi
        print(f"Début du scraping - {total_products} produits à traiter au total")
        print(f"Nombre de catégories : {len(updated_text_content)}")
        print("Les données seront sauvegardées dans:", scraper.csv_filename)
        
        # Parcours des catégories et leurs URLs
        for category, urls in updated_text_content.items():
            print(f"\n=== Traitement de la catégorie : {category} ===")
            print(f"Nombre de produits dans cette catégorie : {len(urls)}")
            
            # Traitement des URLs de la catégorie
            for index, url in enumerate(urls, 1):
                print(f"\nTraitement du produit {index}/{len(urls)} de la catégorie {category}")
                print(f"URL en cours : {url}")
                
                # Traitement du produit avec sa catégorie
                success = scraper.process_product(url, category)
                
                # Mise à jour du compteur et affichage de la progression
                if success:
                    processed_count += 1
                    print(f"Progression globale : {processed_count}/{total_products} produits traités")
                
                # Affichage des statistiques tous les 10 produits
                if processed_count % 10 == 0:
                    print(f"\n=== PROGRESSION ===")
                    print(f"Produits traités avec succès : {processed_count}/{total_products}")
                    print(f"Pourcentage complété : {(processed_count/total_products)*100:.2f}%")
                
                # Petit délai pour éviter la surcharge
                time.sleep(0.5)
                
    except Exception as e:
        print(f"Erreur dans la boucle principale : {e}")
        import traceback
        traceback.print_exc()  # Affiche la trace complète de l'erreur
    
    finally:
        # Résumé final
        print("\n=== Résumé du scraping ===")
        print(f"Nombre total de produits traités : {processed_count}/{total_products}")
        print(f"Taux de réussite : {(processed_count/total_products)*100:.2f}%")
        print(f"Fichier de sauvegarde : {scraper.csv_filename}")
        
        # Affichage des dernières données collectées
        try:
            print("\nDernières entrées enregistrées :")
            df = pd.read_csv(scraper.csv_filename)
            print(df.tail().to_string())
            
            # Statistiques par catégorie
            print("\nStatistiques par catégorie :")
            category_stats = df['Category'].value_counts()
            print(category_stats.to_string())
        except Exception as e:
            print(f"Erreur lors de la lecture du fichier final : {e}")
        
        # Fermeture propre du navigateur
        scraper.driver.quit()
        print("\nTraitement terminé - Navigateur fermé")

if __name__ == "__main__":
    main()